# Headline effects: RQ1

*What is the impact of self-defense techniques on the energy consumption of Android apps?*

Feeds: **§6.2 Headline effects** (Table 4, Table 5, Figure 1), **§6.2 per application and per device**,
and **Threats** (post-charge, composition, linkhub).

**One interval per effect.** The paired t-interval over matched pairs (exact per-n t quantile) is the interval, everywhere.
The application-clustered bootstrap appears only in the labelled REPO file `robustness_app_clustered_bootstrap.csv`, for
Bangcle and freeRASP, never beside a primary interval. It is not computed for Allatori: with five applications, a bootstrap
over clusters has too few to resample.

**p-values** come from the stratified permutation test (6,000 permutations). When no permutation is as extreme as the
observed effect the test returns its floor, 1/6001; that is not a measurement and is reported as **p < 0.001**.
α = 0.05; the six headline contrasts are not multiplicity-corrected (a Holm correction would change none). The plan and protocol written before data collection fixed each technique-against-control comparison and a warm-up/steady split of each cell's energy, which the post-launch and workload windows follow; the pooled test and the paired interval were chosen after data collection.

In [1]:
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import thesis_lib as T

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
REF = T.reference()
for d in (T.TABLES, T.FIGURES, T.SUPERVISOR):
    d.mkdir(exist_ok=True)
T.figure_style()
NB = "notebooks/Headline_Effects.ipynb"

In [2]:
raw = T.load_raw()
print(f"loaded {len(raw)} rows from {len(T.RUN_TABLES)} run tables")
df, chain = T.apply_filters(raw)
series = df[df["batch"].isin(T.SERIES)].copy()
print(f"series cells entering the analysis: {len(series)}")

loaded 4800 rows from 4 run tables
series cells entering the analysis: 2647


## Random streams

Three generators spawned from one seed: the headline permutation tests, the bootstrap, and the permutation tests of the
common-subset sensitivity. Spawned children depend only on their index, so the third stream changes neither the first two
nor the headline p-values.
Arms run in a fixed order: **Bangcle, Allatori, freeRASP**.
Permutation p-values agree with the independent implementation to about the second decimal, not exactly, because that
implementation drew permutations and bootstrap from one interleaved stream.

In [3]:
ss = np.random.SeedSequence(T.SEED)
rng_perm, rng_boot, rng_sens = (np.random.default_rng(s) for s in ss.spawn(3))
print("arm order:", [T.name(a) for a in T.ARMS])

arm order: ['Bangcle', 'Allatori', 'freeRASP']


## §6.2: The headline contrasts

In [4]:
rows, robust = [], []
for arm in T.ARMS:
    for w, col in T.WINDOWS.items():
        d_ = T.phase_set(series, w)                     # the passive phase leaves out medtimer
        p = T.pairs(d_, arm, col)
        m, lo, hi, t = T.ci_naive(p["delta_pct"])
        pt = T.permutation_test(d_, arm, col, rng_perm)
        if arm != "allatori_debug":         # five clusters: no clustered bootstrap for Allatori
            clo, chi, le0 = T.ci_app_clustered(p, rng_boot)
        rows.append(dict(technique=T.name(arm), window=w, n_apps=p["app"].nunique(), n_pairs=len(p),
                         delta_pct=m, ci_lo=lo, ci_hi=hi, p=T.fmt_p(pt), p_permutations=pt["permutations"],
                         p_as_extreme=pt["hits"], mde_pct=T.mde(p["delta_pct"]),
                         sd_pair_delta=p["delta_pct"].std(ddof=1), p_value_raw=pt["p"]))
        if arm != "allatori_debug":
            robust.append(dict(technique=T.name(arm), window=w, n_apps=p["app"].nunique(), delta_pct=m,
                               bootstrap_lo=clo, bootstrap_hi=chi, resamples=T.N_BOOTSTRAP, resample_means_le_zero=le0))
        T.record(f"headline.{T.short(arm)}.{w}", delta_pct=m, ci95=[lo, hi], ci_method="paired t over matched pairs",
                 p=T.fmt_p(pt), p_method=T.P_METHOD, p_note=T.p_note(pt), n_pairs=len(p),
                 n_apps=p["app"].nunique(), mde_pct=T.mde(p["delta_pct"]))
head = pd.DataFrame(rows)
head.drop(columns="p_value_raw").to_csv(T.TABLES / "headline.csv", index=False)
T.register("tables/headline.csv", "REPO", "6.2 RQ1 (headline)", "six headline contrasts, machine-readable: delta-E, paired t CI, permutation p, n, MDE", NB)
rob = pd.DataFrame(robust)
rob.to_csv(T.TABLES / "robustness_app_clustered_bootstrap.csv", index=False)
T.register("tables/robustness_app_clustered_bootstrap.csv", "REPO", "6.2 RQ1 (robustness)",
           "ROBUSTNESS ONLY: application-clustered percentile bootstrap, Bangcle and freeRASP (not Allatori: five clusters); not the reported interval", NB)
print(head[["technique", "window", "n_apps", "n_pairs", "delta_pct", "ci_lo", "ci_hi", "p", "p_as_extreme", "mde_pct"]].round(3).to_string(index=False))

technique   window  n_apps  n_pairs  delta_pct  ci_lo  ci_hi       p  p_as_extreme  mde_pct
  Bangcle workload       8       69     -0.019 -0.813  0.776   0.974          5841    1.132
  Bangcle  startup       8       69      0.098 -0.854  1.050   0.873          5239    1.356
 Allatori workload       5       45     -0.436 -1.521  0.649   0.430          2580    1.543
 Allatori  startup       4       36     -0.294 -1.825  1.236   0.722          4334    2.172
 freeRASP workload       9       78      4.062  2.647  5.476 < 0.001             0    2.016
 freeRASP  startup       8       69     31.791 28.051 35.530 < 0.001             0    5.327


In [5]:
print("delta-E, interval, MDE, pair counts and p against the independent implementation:")
KEY = {v: k for k, v in T.DISPLAY.items()}
for r in head.itertuples():
    k = KEY[r.technique]
    ref = REF["headline_pooled_series"][f"{k}_{r.window}"]
    T.check(f"{r.technique:<9}{r.window:<9} delta-E %", r.delta_pct, ref["delta_pct"])
    T.check(f"{r.technique:<9}{r.window:<9} CI low   ", r.ci_lo, ref["ci_naive"][0], tol=0.01)
    T.check(f"{r.technique:<9}{r.window:<9} CI high  ", r.ci_hi, ref["ci_naive"][1], tol=0.01)
    T.check(f"{r.technique:<9}{r.window:<9} n pairs  ", r.n_pairs, ref["n_pairs"], tol=0)
    T.check(f"{r.technique:<9}{r.window:<9} MDE      ", r.mde_pct, REF["minimum_detectable_effect"][f"{k}_{r.window}"]["mde_pct"])
    if r.p_value_raw < 0.001:   # both at the test's floor: report the agreement, not the floor value
        T.check(f"{r.technique:<9}{r.window:<9} perm p (< 0.001)", r.p_value_raw, ref["permutation_p"], tol=0.02, show=False)
    else:
        T.check(f"{r.technique:<9}{r.window:<9} perm p   ", r.p_value_raw, ref["permutation_p"], tol=0.02)

delta-E, interval, MDE, pair counts and p against the independent implementation:
  Bangcle  workload  delta-E %: -0.0187   (expected -0.0187)  OK
  Bangcle  workload  CI low   : -0.8135   (expected -0.8135)  OK
  Bangcle  workload  CI high  : +0.7760   (expected +0.7760)  OK
  Bangcle  workload  n pairs  : +69.0000   (expected +69.0000)  OK
  Bangcle  workload  MDE      : +1.1320   (expected +1.1320)  OK
  Bangcle  workload  perm p   : +0.9735   (expected +0.9693)  OK
  Bangcle  startup   delta-E %: +0.0977   (expected +0.0977)  OK
  Bangcle  startup   CI low   : -0.8543   (expected -0.8543)  OK
  Bangcle  startup   CI high  : +1.0497   (expected +1.0497)  OK
  Bangcle  startup   n pairs  : +69.0000   (expected +69.0000)  OK
  Bangcle  startup   MDE      : +1.3561   (expected +1.3561)  OK
  Bangcle  startup   perm p   : +0.8732   (expected +0.8690)  OK
  Allatori workload  delta-E %: -0.4364   (expected -0.4364)  OK
  Allatori workload  CI low   : -1.5213   (expected -1.5213)  OK
  Al

### Allatori at five applications

The paired t-interval, the permutation p-value and the minimum detectable effect; no clustered bootstrap (five clusters)
and no count of applications by sign.

In [6]:
for w in ("workload", "startup"):
    r = head.query("technique == 'Allatori' and window == @w").iloc[0]
    print(f"Allatori {w:8s}: {r.delta_pct:+.2f}% (paired t {T.fmt_ci(r.ci_lo, r.ci_hi)}, p = {r.p}), "
          f"{r.n_apps} applications, {r.n_pairs} pairs; MDE +/-{r.mde_pct:.2f}%")

Allatori workload: -0.44% (paired t [-1.52, +0.65], p = 0.430), 5 applications, 45 pairs; MDE +/-1.54%
Allatori startup : -0.29% (paired t [-1.82, +1.24], p = 0.722), 4 applications, 36 pairs; MDE +/-2.17%


## §6.2: Common subset, the Allatori arm's applications (sensitivity)

Allatori is estimated on five applications, Bangcle and freeRASP on nine, so the rows of the headline table are not
computed on the same applications. Bangcle and freeRASP recomputed on the Allatori arm's applications, so cross-arm
rows are like for like; the Allatori rows are identical by construction. p-values from the third random stream.

In [7]:
SUBSET = sorted(T.pairs(series, "allatori_debug", "workload_joules")["app"].unique())
print(f"the common subset ({len(SUBSET)}):", SUBSET)
s_sub = series[series["app"].isin(SUBSET)]
rows = []
for arm in T.ARMS:
    for w, col in T.WINDOWS.items():
        s_w = T.phase_set(s_sub, w)                     # the passive phase leaves out medtimer
        p = T.pairs(s_w, arm, col)
        m, lo, hi, _ = T.ci_naive(p["delta_pct"])
        pt = T.permutation_test(s_w, arm, col, rng_sens)
        rows.append(dict(technique=T.name(arm), window=w, n_apps=p["app"].nunique(), n_pairs=len(p),
                         delta_pct=m, ci_lo=lo, ci_hi=hi, p=T.fmt_p(pt), p_as_extreme=pt["hits"], mde_pct=T.mde(p["delta_pct"])))
        T.record(f"common_subset.{T.short(arm)}.{w}", delta_pct=m, ci95=[lo, hi], p=T.fmt_p(pt), p_note=T.p_note(pt),
                 n_pairs=len(p), n_apps=p["app"].nunique(), apps=SUBSET)
sub = pd.DataFrame(rows)
print(sub.round(3).to_string(index=False))

the common subset (5): ['calculator', 'medtimer', 'pdfviewer', 'poetskingdom', 'tipuous']


technique   window  n_apps  n_pairs  delta_pct  ci_lo  ci_hi       p  p_as_extreme  mde_pct
  Bangcle workload       4       36      0.321 -0.753  1.394   0.650          3897    1.524
  Bangcle  startup       4       36      0.350 -0.913  1.613   0.654          3924    1.792
 Allatori workload       5       45     -0.436 -1.521  0.649   0.443          2659    1.543
 Allatori  startup       4       36     -0.294 -1.825  1.236   0.721          4327    2.172
 freeRASP workload       5       45      4.379  2.620  6.138 < 0.001             0    2.500
 freeRASP  startup       4       36     28.175 23.857 32.493 < 0.001             0    6.130


## Table 4: main results

Block 1: the six headline contrasts (primary). Block 2: Bangcle and freeRASP on the Allatori arm's applications.

In [8]:
def fmt_rows(d, interval=("ci_lo", "ci_hi")):
    return pd.DataFrame({"Technique": d["technique"], "Window": d["window"].map(T.WINDOW_DISPLAY),
                         "Apps": d["n_apps"], "Pairs": d["n_pairs"],
                         "dE (%)": d["delta_pct"].map(lambda x: f"{x:+.2f}"),
                         "95% CI": [T.fmt_ci(a, b) for a, b in zip(d[interval[0]], d[interval[1]])],
                         "p": d["p"], "MDE (%)": d["mde_pct"].map(lambda x: f"{x:.2f}")})
b1 = fmt_rows(head)
b2 = fmt_rows(sub[sub["technique"] != "Allatori"])
t4 = pd.concat([b1, b2], ignore_index=True)
blocks = {0: "Primary: every application in each technique's arm",
          len(b1): f"Bangcle and freeRASP on the {len(SUBSET)} Allatori applications (sensitivity)"}
t4.insert(0, "Block", [blocks[max(k for k in blocks if k <= i)] for i in range(len(t4))])
paths = T.write_tex(t4.drop(columns="Block").rename(columns={"dE (%)": "ΔE (%)"}), "table4_main_results",
    caption="Energy effect of each technique against its control (matched pairs, three series batches pooled).",
    label="tab:main-results", colfmt="llrrrlll", blocks=blocks,
    notes="ΔE: mean over matched pairs of the percentage difference from the control. 95% CI: paired t-interval over pairs. "
          "p: stratified permutation test, 6,000 permutations; < 0.001 means none was as extreme as observed. "
          "MDE: minimum detectable effect at 80% power, α = 0.05. Passive phase: medtimer left out in every build.")
t4.to_csv(T.TABLES / "table4_main_results.csv", index=False)   # the CSV keeps the Block column
T.register(paths, "REPO", "6.2 RQ1 (headline)", "main results: n apps, delta-E, paired t CI, p, MDE; common-subset block", NB)
print(t4.to_string(index=False))

                                                            Block Technique  Window  Apps  Pairs dE (%)           95% CI       p MDE (%)
               Primary: every application in each technique's arm   Bangcle  Active     8     69  -0.02   [-0.81, +0.78]   0.974    1.13
               Primary: every application in each technique's arm   Bangcle Passive     8     69  +0.10   [-0.85, +1.05]   0.873    1.36
               Primary: every application in each technique's arm  Allatori  Active     5     45  -0.44   [-1.52, +0.65]   0.430    1.54
               Primary: every application in each technique's arm  Allatori Passive     4     36  -0.29   [-1.82, +1.24]   0.722    2.17
               Primary: every application in each technique's arm  freeRASP  Active     9     78  +4.06   [+2.65, +5.48] < 0.001    2.02
               Primary: every application in each technique's arm  freeRASP Passive     8     69 +31.79 [+28.05, +35.53] < 0.001    5.33
Bangcle and freeRASP on the 5 Allatori ap

## Minimum detectable effect (repository table)

MDE = (t₀.₉₇₅ + t₀.₈₀) · s / √n over matched pairs, per arm and window: the smallest true effect this design
would detect with 80% power at α = 0.05. It is computed from the observed variance, so it describes this dataset's
resolution; a null below it is "not detected", not "absent". It is the MDE column of Table 4; this file adds the SD and pair count behind it.

In [9]:
t6 = pd.DataFrame({"Technique": head["technique"], "Window": head["window"].map(T.WINDOW_DISPLAY),
                   "Pairs": head["n_pairs"], "SD of pair ΔE (pp)": head["sd_pair_delta"].map(lambda x: f"{x:.2f}"),
                   "MDE (±%)": head["mde_pct"].map(lambda x: f"{x:.2f}"),
                   "Observed ΔE (%)": head["delta_pct"].map(lambda x: f"{x:+.2f}")})
t6.to_csv(T.TABLES / "minimum_detectable_effect.csv", index=False)
T.register("tables/minimum_detectable_effect.csv", "REPO", "6.2 RQ1 / Threats (power)",
           "MDE per arm and window, with pairs, SD and the observed effect (the MDE column of Table 4, with its inputs)", NB)
for r in head.itertuples():
    T.record(f"mde.{KEY[r.technique]}.{r.window}", mde_pct=r.mde_pct, n_pairs=r.n_pairs)
print(t6.to_string(index=False))

Technique  Window  Pairs SD of pair ΔE (pp) MDE (±%) Observed ΔE (%)
  Bangcle  Active     69               3.31     1.13           -0.02
  Bangcle Passive     69               3.96     1.36           +0.10
 Allatori  Active     45               3.61     1.54           -0.44
 Allatori Passive     36               4.52     2.17           -0.29
 freeRASP  Active     78               6.27     2.02           +4.06
 freeRASP Passive     69              15.57     5.33          +31.79


## Table 5: Allatori applicability

What happened when Allatori was applied to each of the nine applications. These are findings about how the tool behaves on
real applications, and only that: which applications enter the Allatori estimate is in Table 1. (itsok's Allatori build,
excluded there by the build-type rule, is a process error of this work, not a property of the tool; its row says so.) Renaming counts are from the swept APKs (`tools/verify_allatori_obfuscation.py --swept`); cell counts and
crash signatures are from the run tables, the three series batches (the pilot enters no Results number).

In [10]:
ren = pd.read_csv(T.DATA / "inputs" / "allatori_renaming.csv").set_index("app")
apps = sorted(series["app"].unique())
r_all = raw[(raw["variant"] == "allatori_debug") & raw["batch"].isin(T.SERIES)].copy()   # series only
sig = pd.concat(pd.read_csv(T.RUN_TABLES[b], usecols=["app", "variant", "app_crash_signature"], dtype=str) for b in T.SERIES)
sig = sig[sig["variant"] == "allatori_debug"]
rows = []
for app in apps:
    c = r_all[r_all["app"] == app]
    valid = int((c["scientifically_valid"] == "True").sum())
    renamed = int(ren.loc[app, "classes_only_in_allatori"])
    ame = int(sig.loc[sig["app"] == app, "app_crash_signature"].fillna("").str.contains("AbstractMethodError").sum())
    if ren.loc[app, "verdict"] == "OBFUSCATED":
        outcome = ("obfuscated, runs" if valid > len(c) / 2 else
                   f"obfuscated, crashes at start-up (AbstractMethodError in {ame} of {len(c)} cells)")
        if (app, "allatori_debug") in T.EXCLUDED_BUILDS:
            outcome += "; release build, not in the arm"
    elif ren.loc[app, "verdict"] == "IDENTICAL_CLASS_SET":
        outcome = "runs; not transformed (class set identical to the control)"
    else:
        outcome = f"runs; not transformed ({int(ren.loc[app, 'classes_only_in_baseline'])} unreachable classes only in its control)"
    rows.append({"App": app, "Classes renamed": renamed, "Outcome": outcome, "Valid cells": f"{valid} / {len(c)}"})
t5 = pd.DataFrame(rows)
print(t5.to_string(index=False))
paths = T.write_tex(t5, "table5_allatori_applicability",
    caption="Applying Allatori to the nine applications: renaming and run-time outcome. Bold: the values the thesis states in words.",
    label="tab:allatori", colfmt="lrp{7cm}r", star=True,
    bold_cells={(int(t5.index[t5["App"] == a][0]), j) for a in t5.loc[t5["Outcome"].str.contains("crashes"), "App"] for j in (2, 3)},
    notes=f"Valid cells: the three series batches; keep_recipe's {int(((r_all.app == 'keep_recipe') & (r_all.phone == 'pixel3')).sum())} Pixel 3 cells are excluded for every build (the device cannot run it). keep_recipe's and linkhub's obfuscated builds were not transformed, so they are not in the arm. "
          "itsok's obfuscated build is a release "
          "build and is not part of the obfuscation arm.")
T.register(paths, "SUPPLEMENT", "Supplement (Allatori applicability)", "Allatori on nine apps: classes renamed, run-time outcome, valid cells (the tool's behaviour only)", NB)
t5.to_csv(T.TABLES / "allatori_applicability_detail.csv", index=False)
T.register("tables/allatori_applicability_detail.csv", "REPO", "6.2 RQ1 (Allatori applicability)",
           "the Allatori applicability table: classes renamed, run-time outcome and valid cells per application", NB)
T.record("allatori_applicability", obfuscated_runs=list(t5.loc[t5["Outcome"].str.startswith("obfuscated, runs"), "App"]),
         obfuscated_crashes=list(t5.loc[t5["Outcome"].str.contains("crashes"), "App"]),
         renames_nothing=list(t5.loc[t5["Outcome"].str.startswith("runs;"), "App"]),
         documenter_valid_cells=t5.loc[t5["App"] == "documenter", "Valid cells"].iloc[0])

         App  Classes renamed                                                                 Outcome Valid cells
  calculator               61                                                        obfuscated, runs     89 / 90
  documenter              143 obfuscated, crashes at start-up (AbstractMethodError in 90 of 90 cells)      0 / 90
       itsok              124                         obfuscated, runs; release build, not in the arm     88 / 90
 keep_recipe                0              runs; not transformed (class set identical to the control)     60 / 90
     linkhub                0      runs; not transformed (52 unreachable classes only in its control)     89 / 90
    medtimer              400                                                        obfuscated, runs     89 / 90
   pdfviewer               25                                                        obfuscated, runs     82 / 90
poetskingdom              490                                                        obf

## §6.2: Per batch (workload)

Pairs formed within one batch, so `batch` leaves the group key.

In [11]:
rows = []
for b in ("pilot",) + T.SERIES:
    d = df[df["batch"] == b]
    for arm in T.ARMS:
        p = T.pairs(d, arm, "workload_joules", by_batch=False)
        m, lo, hi, _ = T.ci_naive(p["delta_pct"])
        rows.append(dict(batch=b, technique=T.name(arm), n_pairs=len(p), delta_pct=m, ci_lo=lo, ci_hi=hi))
        T.check(f"{b:<8}{T.name(arm):<9}", m, REF["per_batch_workload"][f"{b}_{T.short(arm)}"]["delta_pct"])
pd.DataFrame(rows).to_csv(T.TABLES / "per_batch.csv", index=False)
T.register("tables/per_batch.csv", "REPO", "6.2 RQ1 (per batch)", "active-phase delta-E per batch and technique", NB)

  pilot   Bangcle  : +5.0993   (expected +5.0993)  OK
  pilot   Allatori : -0.2364   (expected -0.2364)  OK
  pilot   freeRASP : +9.3313   (expected +9.3313)  OK
  series1 Bangcle  : +0.5864   (expected +0.5864)  OK
  series1 Allatori : -1.0481   (expected -1.0481)  OK
  series1 freeRASP : +4.1975   (expected +4.1975)  OK
  series2 Bangcle  : -0.4771   (expected -0.4771)  OK


  series2 Allatori : -0.5095   (expected -0.5095)  OK
  series2 freeRASP : +3.8145   (expected +3.8145)  OK
  series3 Bangcle  : -0.1655   (expected -0.1655)  OK
  series3 Allatori : +0.2485   (expected +0.2485)  OK


  series3 freeRASP : +4.1729   (expected +4.1729)  OK


## §6.2 (RQ1): Per device, freeRASP, with the post-charge sensitivity

No cell is excluded on the post-charge flag (decision 2026-09-28). The Pixel 9 row carries **both** values because the
unfiltered figure is biased toward the null on that contrast; the pooled sensitivity below shows the headline is insensitive.

In [12]:
flag = T.post_charge_flag(series)
s_f = series[~flag]
print(f"post-charge flagged: {int(flag.sum())} of {len(series)} cells (K = {T.K_PHONE})")
rows = []
for phone in ("pixel3", "pixel6", "pixel9"):
    p0 = T.pairs(series[series["phone"] == phone], "freerasp_debug", "workload_joules")
    p1 = T.pairs(s_f[s_f["phone"] == phone], "freerasp_debug", "workload_joules")
    m0, lo0, hi0, _ = T.ci_naive(p0["delta_pct"]); m1, lo1, hi1, _ = T.ci_naive(p1["delta_pct"])
    rows.append(dict(phone=T.PHONE_DISPLAY[phone], n_pairs=len(p0), delta_pct=m0, ci_lo=lo0, ci_hi=hi0,
                     filtered_n_pairs=len(p1), filtered_delta_pct=m1, filtered_ci_lo=lo1, filtered_ci_hi=hi1))
    ref = REF["post_charge_sensitivity"]["per_phone_freerasp_workload"][phone]
    T.check(f"{phone} unfiltered", m0, ref["unfiltered_pct"]); T.check(f"{phone} filtered  ", m1, ref["filtered_pct"])
    T.record(f"per_phone.freerasp.workload.{phone}", delta_pct=m0, ci95=[lo0, hi0], n_pairs=len(p0),
             post_charge_filtered_delta_pct=m1, post_charge_filtered_ci95=[lo1, hi1], note="unfiltered is primary")
pd.DataFrame(rows).to_csv(T.TABLES / "per_phone_freerasp.csv", index=False)
T.register("tables/per_phone_freerasp.csv", "REPO", "6.2 RQ1 (per device)", "freeRASP active-phase delta-E per phone, unfiltered (primary) and post-charge filtered", NB)
print(pd.DataFrame(rows).round(2).to_string(index=False))

post-charge flagged: 451 of 2647 cells (K = {'pixel3': 2, 'pixel6': 6, 'pixel9': 4})
  pixel3 unfiltered: +8.0402   (expected +8.0402)  OK
  pixel3 filtered  : +8.3173   (expected +8.3173)  OK
  pixel6 unfiltered: +4.0445   (expected +4.0445)  OK
  pixel6 filtered  : +3.1278   (expected +3.1278)  OK
  pixel9 unfiltered: +0.5421   (expected +0.5421)  OK
  pixel9 filtered  : +1.7510   (expected +1.7510)  OK
  phone  n_pairs  delta_pct  ci_lo  ci_hi  filtered_n_pairs  filtered_delta_pct  filtered_ci_lo  filtered_ci_hi
Pixel 3       24       8.04   5.41  10.67                24                8.32            5.64           10.99
Pixel 6       27       4.04   2.48   5.61                27                3.13            1.54            4.72
Pixel 9       27       0.54  -1.93   3.01                27                1.75            0.11            3.39


In [13]:
rows = []
for arm in T.ARMS:
    for w, col in T.WINDOWS.items():
        u = T.delta_e(T.pairs(T.phase_set(series, w), arm, col)); f = T.delta_e(T.pairs(T.phase_set(s_f, w), arm, col))
        rows.append(dict(technique=T.name(arm), window=w, unfiltered=u, filtered=f, shift_pp=f - u))
pcs = pd.DataFrame(rows)
pcs.to_csv(T.TABLES / "post_charge_pooled_sensitivity.csv", index=False)
T.register("tables/post_charge_pooled_sensitivity.csv", "REPO", "Threats (post-charge)", "pooled headline with and without post-charge cells", NB)
print(pcs.round(3).to_string(index=False))
print(f"\nlargest shift in any pooled headline: {pcs['shift_pp'].abs().max():.2f} pp")
T.record("post_charge.max_pooled_shift_pp", value=pcs["shift_pp"].abs().max())

technique   window  unfiltered  filtered  shift_pp
  Bangcle workload      -0.019     0.231     0.249
  Bangcle  startup       0.098     0.235     0.137
 Allatori workload      -0.436     0.344     0.780
 Allatori  startup      -0.294     0.472     0.767
 freeRASP workload       4.062     4.248     0.186
 freeRASP  startup      31.791    31.933     0.143

largest shift in any pooled headline: 0.78 pp


## §6.2 (RQ1): Per application, with nulls marked

In [14]:
NOTES = {"documenter": {"allatori": "no working build"}, "keep_recipe": {"allatori": "negative control"},
         "linkhub": {"allatori": "negative control"}, "pdfviewer": {"allatori": ""},
         "itsok": {"allatori": "excluded: release build"}}
rows = []
for arm in T.ARMS:
    p = T.pairs(series, arm, "workload_joules")
    for app, g in p.groupby("app"):
        rows.append(dict(app=app, technique=T.name(arm), n_pairs=len(g), delta_pct=g["delta_pct"].mean(),
                         note=NOTES.get(app, {}).get(T.short(arm), "")))
        T.check(f"{app:<13}{T.name(arm):<9}", g["delta_pct"].mean(), REF["per_app_workload"][app][T.short(arm)]["delta_pct"])
for app, n in NOTES.items():
    for tech, note in n.items():
        if not any(r["app"] == app and r["technique"] == T.DISPLAY[tech] for r in rows):
            rows.append(dict(app=app, technique=T.DISPLAY[tech], n_pairs=0, delta_pct=np.nan, note=note))
pa = pd.DataFrame(rows)
pa.to_csv(T.TABLES / "per_app.csv", index=False)
T.register("tables/per_app.csv", "REPO", "6.2 RQ1 (per application)", "active-phase delta-E per application and technique, nulls marked", NB)
print(pa.pivot(index="app", columns="technique", values="delta_pct").round(2).to_string())

  calculator   Bangcle  : +1.1040   (expected +1.1040)  OK
  documenter   Bangcle  : +1.2134   (expected +1.2134)  OK
  itsok        Bangcle  : -1.1172   (expected -1.1172)  OK
  keep_recipe  Bangcle  : -2.0397   (expected -2.0397)  OK
  linkhub      Bangcle  : -0.1625   (expected -0.1625)  OK
  pdfviewer    Bangcle  : -1.2235   (expected -1.2235)  OK
  poetskingdom Bangcle  : +0.8754   (expected +0.8754)  OK
  tipuous      Bangcle  : +0.5266   (expected +0.5266)  OK
  calculator   Allatori : -0.4319   (expected -0.4319)  OK
  medtimer     Allatori : -0.1572   (expected -0.1572)  OK
  pdfviewer    Allatori : -0.3712   (expected -0.3712)  OK
  poetskingdom Allatori : -0.3124   (expected -0.3124)  OK
  tipuous      Allatori : -0.9091   (expected -0.9091)  OK


  calculator   freeRASP : +3.0961   (expected +3.0961)  OK
  documenter   freeRASP : +9.1437   (expected +9.1437)  OK
  itsok        freeRASP : +2.6619   (expected +2.6619)  OK
  keep_recipe  freeRASP : +2.5266   (expected +2.5266)  OK
  linkhub      freeRASP : -0.1846   (expected -0.1846)  OK
  medtimer     freeRASP : +4.1974   (expected +4.1974)  OK
  pdfviewer    freeRASP : +0.3826   (expected +0.3826)  OK
  poetskingdom freeRASP : +10.4557   (expected +10.4557)  OK
  tipuous      freeRASP : +3.7632   (expected +3.7632)  OK
technique     Allatori  Bangcle  freeRASP
app                                      
calculator       -0.43     1.10      3.10
documenter         NaN     1.21      9.14
itsok              NaN    -1.12      2.66
keep_recipe        NaN    -2.04      2.53
linkhub            NaN    -0.16     -0.18
medtimer         -0.16      NaN      4.20
pdfviewer        -0.37    -1.22      0.38
poetskingdom     -0.31     0.88     10.46
tipuous          -0.91     0.53      3.76


## Threats: obfuscation-arm composition, leave-one-application-out, and linkhub

In [15]:
rows = []
a = T.pairs(series, "allatori_debug", "workload_joules")
df_inc, _ = T.apply_filters(raw, exclude_builds=())   # reference only: with the excluded release build
a_inc = T.pairs(df_inc[df_inc["batch"].isin(T.SERIES)], "allatori_debug", "workload_joules")
for label, x in (("the Allatori arm", a), ("without pdfviewer", a[a["app"] != "pdfviewer"]),
                 ("reference only: with itsok's release build (excluded by rule)", a_inc)):
    m, lo, hi, _ = T.ci_naive(x["delta_pct"])
    rows.append(dict(composition=label, n_apps=x["app"].nunique(), n_pairs=len(x), delta_pct=m, ci_lo=lo, ci_hi=hi))
    T.record(f"allatori_composition.{('with_itsok_release_build' if label.startswith('reference') else label.split(' (')[0].replace(' ', '_'))}",
             delta_pct=m, ci95=[lo, hi], n_pairs=len(x), n_apps=x["app"].nunique())
comp = pd.DataFrame(rows); comp.to_csv(T.TABLES / "allatori_composition.csv", index=False)
T.register("tables/allatori_composition.csv", "REPO", "Threats (Allatori arm composition)",
           "Allatori workload: the arm, without pdfviewer, and for reference with itsok's excluded release build", NB)
print(comp.round(2).to_string(index=False))
rows = []
for arm in T.ARMS:
    for w, col in T.WINDOWS.items():
        p = T.pairs(T.phase_set(series, w), arm, col)
        _lo_full = T.ci_naive(p["delta_pct"])
        _lo_detected = _lo_full[1] > 0 or _lo_full[2] < 0
        loo = {app: T.ci_naive(p.loc[p["app"] != app, "delta_pct"])[:3] for app in p["app"].unique()}
        _lo_min = min(loo, key=lambda a: loo[a][0]); _lo_max = max(loo, key=lambda a: loo[a][0])
        rows.append(dict(technique=T.name(arm), window=w, n_apps=p["app"].nunique(),
                         loo_min=loo[_lo_min][0], loo_max=loo[_lo_max][0],
                         dropped_for_min=_lo_min, ci_lo_for_min=loo[_lo_min][1], ci_hi_for_min=loo[_lo_min][2],
                         dropped_for_max=_lo_max, ci_lo_for_max=loo[_lo_max][1], ci_hi_for_max=loo[_lo_max][2],
                         same_conclusion_for_every_drop=all((v[1] > 0 or v[2] < 0) == _lo_detected for v in loo.values())))
loo = pd.DataFrame(rows); loo.to_csv(T.TABLES / "leave_one_app_out.csv", index=False)
T.register("tables/leave_one_app_out.csv", "REPO", "6.2 RQ1 / Threats (leave one application out)",
           "range of delta-E when each application is left out in turn: the application behind each end, its 95% CI, and whether any drop changes the conclusion", NB)
print(loo.round(2).to_string(index=False))

                                                  composition  n_apps  n_pairs  delta_pct  ci_lo  ci_hi
                                             the Allatori arm       5       45      -0.44  -1.52   0.65
                                            without pdfviewer       4       36      -0.45  -1.64   0.73
reference only: with itsok's release build (excluded by rule)       6       54      -0.67  -1.70   0.35
technique   window  n_apps  loo_min  loo_max dropped_for_min  ci_lo_for_min  ci_hi_for_min dropped_for_max  ci_lo_for_max  ci_hi_for_max  same_conclusion_for_every_drop
  Bangcle workload       8    -0.20     0.17      documenter          -1.10           0.70     keep_recipe          -0.63           0.98                            True
  Bangcle  startup       8    -0.11     0.23      calculator          -1.15           0.94       pdfviewer          -0.80           1.26                            True
 Allatori workload       5    -0.51    -0.32        medtimer          -1.77  

### linkhub left out (Threats)

linkhub's control carries 44 obfuscated-looking classes its freeRASP build lacks (Descriptive_Statistics, control-screen note).
linkhub is kept in the cohort, because removing an application after seeing results would be an exclusion by result; this
shows what removing it would change.

In [16]:
rows = []
for arm in ("bangcle_debug", "freerasp_debug"):
    for w, col in T.WINDOWS.items():
        for label, d in (("all applications", series), ("without linkhub", series[series["app"] != "linkhub"])):
            p = T.pairs(T.phase_set(d, w), arm, col)
            m, lo, hi, _ = T.ci_naive(p["delta_pct"])
            rows.append(dict(technique=T.name(arm), window=w, cohort=label, n_pairs=len(p), delta_pct=m, ci_lo=lo, ci_hi=hi))
lh = pd.DataFrame(rows)
lh.to_csv(T.TABLES / "linkhub_leave_out.csv", index=False)
T.register("tables/linkhub_leave_out.csv", "REPO", "Threats (linkhub control residue)", "Bangcle and freeRASP with and without linkhub", NB)
print(lh.round(2).to_string(index=False))
for r in lh[lh["cohort"] == "without linkhub"].itertuples():
    T.record(f"linkhub_left_out.{KEY[r.technique]}.{r.window}", delta_pct=r.delta_pct, ci95=[r.ci_lo, r.ci_hi], n_pairs=r.n_pairs)

technique   window           cohort  n_pairs  delta_pct  ci_lo  ci_hi
  Bangcle workload all applications       69      -0.02  -0.81   0.78
  Bangcle workload  without linkhub       60       0.00  -0.87   0.88
  Bangcle  startup all applications       69       0.10  -0.85   1.05
  Bangcle  startup  without linkhub       60       0.14  -0.84   1.12
 freeRASP workload all applications       78       4.06   2.65   5.48
 freeRASP workload  without linkhub       69       4.62   3.15   6.08
 freeRASP  startup all applications       69      31.79  28.05  35.53
 freeRASP  startup  without linkhub       60      33.34  29.25  37.43


### Supervisor notes only: ten applications

Written to `tables/supervisor_only/`. No ten-application figure appears in the thesis.

In [17]:
df10, _ = T.apply_filters(raw, exclude_apps=())
s10 = df10[df10["batch"].isin(T.SERIES)]
REF10 = json.loads((T.DATA / "derived" / "thesis_numbers_ten_apps_sensitivity.json").read_text())
rows = []
for arm in T.ARMS:
    for w, col in T.WINDOWS.items():
        m10 = T.delta_e(T.pairs(T.phase_set(s10, w), arm, col)); m9 = head.loc[(head["technique"] == T.name(arm)) & (head["window"] == w), "delta_pct"].iloc[0]
        rows.append(dict(technique=T.name(arm), window=w, ten_apps=m10, nine_apps=m9, change_pp=m9 - m10))
        T.check(f"ten-app {T.name(arm):<9}{w:<9}", m10, REF10["headline_pooled_series"][f"{T.short(arm)}_{w}"]["delta_pct"])
pd.DataFrame(rows).to_csv(T.SUPERVISOR / "sensitivity_ten_vs_nine_apps.csv", index=False)
T.register("tables/supervisor_only/sensitivity_ten_vs_nine_apps.csv", "SUPERVISOR", "none",
           "headline delta-E with and without androidskk; not cited in the thesis", NB)
print(pd.DataFrame(rows).round(3).to_string(index=False))

  ten-app Bangcle  workload : +0.2170   (expected +0.2170)  OK
  ten-app Bangcle  startup  : +0.1196   (expected +0.1196)  OK
  ten-app Allatori workload : +0.3470   (expected +0.3470)  OK
  ten-app Allatori startup  : +0.3407   (expected +0.3407)  OK
  ten-app freeRASP workload : +4.3212   (expected +4.3212)  OK
  ten-app freeRASP startup  : +32.7004   (expected +32.7004)  OK
technique   window  ten_apps  nine_apps  change_pp
  Bangcle workload     0.217     -0.019     -0.236
  Bangcle  startup     0.120      0.098     -0.022
 Allatori workload     0.347     -0.436     -0.783
 Allatori  startup     0.341     -0.294     -0.635
 freeRASP workload     4.321      4.062     -0.260
 freeRASP  startup    32.700     31.791     -0.910


## Every matched pair (for Figure 1), and whether any single pair drives a headline

`pairs_delta.csv` holds every matched pair's ΔE, with the number of valid cells and the within-group CV on each side.
Dropping each pair in turn shows how far any one pair moves the estimate. `single_pair_influence.csv` names the pair
behind each end of that range, gives the 95% paired t-interval once it is dropped, and says whether dropping any single
pair would change the interval's position relative to zero (contains it, or excludes it).

In [18]:
rows, infl = [], []
keys = ["app", "phone", "batch"]
cells = {}                                              # per window: valid cells and within-group CV of every group
for w, col in T.WINDOWS.items():
    g = series[series[col] > 0].groupby(keys + ["variant"])[col].agg(["count", "mean", "std"])
    cells[w] = g.assign(cv_pct=100 * g["std"] / g["mean"]).reset_index()
side = lambda g, v, tag: g.loc[g["variant"] == v, keys + ["count", "cv_pct"]].rename(
    columns={"count": f"n_cells_{tag}", "cv_pct": f"cv_{tag}_pct"})
excludes_zero = lambda lo, hi: bool(lo > 0 or hi < 0)
for arm in T.ARMS:
    for w, col in T.WINDOWS.items():
        p = T.pairs(T.phase_set(series, w), arm, col).assign(technique=T.name(arm), window=w)
        n0 = len(p)
        p = p.merge(side(cells[w], "baseline_debug", "baseline"), on=keys).merge(side(cells[w], arm, "treated"), on=keys)
        assert len(p) == n0, "a pair lost its cell summary"
        rows.append(p[["technique", "window", "app", "phone", "batch", "baseline", "treated", "delta_pct",
                       "n_cells_baseline", "n_cells_treated", "cv_baseline_pct", "cv_treated_pct"]])
        x = p["delta_pct"].to_numpy()
        m, lo, hi, _ = T.ci_naive(x)
        drops = [T.ci_naive(np.delete(x, i)) for i in range(len(x))]     # (mean, lo, hi, t) without pair i
        loo = [d[0] for d in drops]
        i_min, i_max = int(np.argmin(loo)), int(np.argmax(loo))
        assert i_min == int(np.argmax(x)) and i_max == int(np.argmin(x))  # dropping the largest pair gives the lowest mean
        pid = lambda i: " / ".join(p[k].iloc[i] for k in keys)
        same = all(excludes_zero(d[1], d[2]) == excludes_zero(lo, hi) for d in drops)
        infl.append(dict(technique=T.name(arm), window=w, n_pairs=len(p), delta_pct=m, ci_lo=lo, ci_hi=hi,
                         drop_one_pair_min=loo[i_min], dropped_for_min=pid(i_min), dropped_for_min_delta_pct=x[i_min],
                         ci_lo_for_min=drops[i_min][1], ci_hi_for_min=drops[i_min][2],
                         drop_one_pair_max=loo[i_max], dropped_for_max=pid(i_max), dropped_for_max_delta_pct=x[i_max],
                         ci_lo_for_max=drops[i_max][1], ci_hi_for_max=drops[i_max][2],
                         same_conclusion_for_every_drop=same))
        T.record(f"single_pair_influence.{T.short(arm)}.{w}", drop_one_pair_range=[loo[i_min], loo[i_max]],
                 dropped_for_min=pid(i_min), dropped_for_min_delta_pct=x[i_min],
                 ci95_for_min=[drops[i_min][1], drops[i_min][2]],
                 dropped_for_max=pid(i_max), dropped_for_max_delta_pct=x[i_max],
                 ci95_for_max=[drops[i_max][1], drops[i_max][2]], same_conclusion_for_every_drop=same)
pd.concat(rows).to_csv(T.TABLES / "pairs_delta.csv", index=False)
T.register("tables/pairs_delta.csv", "REPO", "6.2 RQ1 (headline)",
           "every matched pair's delta-E, with valid cells and within-group CV on each side: the points behind Figure 1 and Table 4", NB)
inf = pd.DataFrame(infl); inf.to_csv(T.TABLES / "single_pair_influence.csv", index=False)
T.register("tables/single_pair_influence.csv", "REPO", "6.2 RQ1 / Threats",
           "range of each headline when any one pair is dropped; the pair behind each end and the 95% CI without it", NB)
with pd.option_context("display.width", 200):
    print(inf[["technique", "window", "n_pairs", "delta_pct", "drop_one_pair_min", "drop_one_pair_max",
               "same_conclusion_for_every_drop"]].round(2).to_string(index=False))
    print()
    print(inf[["technique", "window", "dropped_for_min", "dropped_for_min_delta_pct", "ci_lo_for_min", "ci_hi_for_min"]]
          .round(2).to_string(index=False))

technique   window  n_pairs  delta_pct  drop_one_pair_min  drop_one_pair_max  same_conclusion_for_every_drop
  Bangcle workload       69      -0.02              -0.15               0.17                            True
  Bangcle  startup       69       0.10              -0.06               0.25                            True
 Allatori workload       45      -0.44              -0.72              -0.20                            True
 Allatori  startup       36      -0.29              -0.64               0.02                            True
 freeRASP workload       78       4.06               3.87               4.29                            True
 freeRASP  startup       69      31.79              31.20              32.20                            True

technique   window                 dropped_for_min  dropped_for_min_delta_pct  ci_lo_for_min  ci_hi_for_min
  Bangcle workload poetskingdom / pixel9 / series3                       9.07          -0.91           0.61
  Bangcle  startup  

In [19]:
# Per device, for every technique and window (exploratory). RQ1 promises the per-device view for every technique.
_pd_pr = pd.read_csv(T.TABLES / "pairs_delta.csv")
_pd_rows = []
for (_pd_t, _pd_w, _pd_ph), _pd_g in _pd_pr.groupby(["technique", "window", "phone"]):
    _pd_m, _pd_lo, _pd_hi, _ = T.ci_naive(_pd_g["delta_pct"].to_numpy())
    _pd_rows.append(dict(technique=_pd_t, window=_pd_w, device=T.PHONE_DISPLAY[_pd_ph], n_pairs=len(_pd_g),
                         delta_pct=_pd_m, ci_lo=_pd_lo, ci_hi=_pd_hi, excludes_zero=bool(_pd_lo > 0 or _pd_hi < 0)))
_pd = pd.DataFrame(_pd_rows)
_pd_fr = pd.read_csv(T.TABLES / "per_phone_freerasp.csv").set_index("phone")
for _pd_r in _pd[(_pd["technique"] == "freeRASP") & (_pd["window"] == "workload")].itertuples():
    assert abs(_pd_r.delta_pct - _pd_fr.loc[_pd_r.device, "delta_pct"]) < 1e-9, "differs from per_phone_freerasp.csv"
_pd.to_csv(T.TABLES / "per_device_effects.csv", index=False)
T.register("tables/per_device_effects.csv", "REPO", "6.2 RQ1 (per device, exploratory)",
           "delta-E per technique, window and device: mean over matched pairs with paired t-interval (exploratory, no p-values; superseded by per_device_tests.csv)", NB)
print(_pd.round(2).to_string(index=False))

technique   window  device  n_pairs  delta_pct  ci_lo  ci_hi  excludes_zero
 Allatori  startup Pixel 3       12       0.68  -1.61   2.98          False
 Allatori  startup Pixel 6       12      -0.53  -1.72   0.66          False
 Allatori  startup Pixel 9       12      -1.04  -5.38   3.31          False
 Allatori workload Pixel 3       15      -0.19  -1.19   0.81          False
 Allatori workload Pixel 6       15       0.09  -0.72   0.90          False
 Allatori workload Pixel 9       15      -1.21  -4.46   2.05          False
  Bangcle  startup Pixel 3       21       0.90  -0.44   2.23          False
  Bangcle  startup Pixel 6       24      -0.02  -1.10   1.06          False
  Bangcle  startup Pixel 9       24      -0.48  -2.85   1.88          False
  Bangcle workload Pixel 3       21       0.75   0.08   1.42           True
  Bangcle workload Pixel 6       24       0.25  -0.61   1.11          False
  Bangcle workload Pixel 9       24      -0.96  -3.06   1.14          False
 freeRASP  s

## §4.3.2: The negative controls, the pipeline's noise floor (exploratory)

Two builds labelled as obfuscated were declared null treatments before the series (keep_recipe and linkhub: the
obfuscator did not run on either). They are excluded from every estimate by the filter chain's
"not a negative control" step. Here they are compared with their own controls exactly as the treated builds are: every other
filter applied, series batches, matched pairs (device, batch), the mean percentage difference with its paired t-interval.
The design reads this difference as the noise floor of the whole pipeline. Exploratory, no p.

In [20]:
_nc_d = T.load_raw()
for _nc_name, _nc_pred in T.filter_steps():
    if _nc_name != "not a negative control":
        _nc_d = _nc_d[_nc_pred(_nc_d)]
_nc_d = _nc_d[_nc_d["batch"].isin(T.SERIES)]
_nc_flag = _nc_d["negative_control"].fillna("").str.strip()
_nc_builds = sorted({(_nc_a, _nc_v) for _nc_a, _nc_v, _nc_f in zip(_nc_d["app"], _nc_d["variant"], _nc_flag) if _nc_f not in ("", "None", "False")})
assert _nc_builds == [("keep_recipe", "allatori_debug"), ("linkhub", "allatori_debug")], _nc_builds
_nc_rows = []
for _nc_scope, _nc_apps in (("keep_recipe", ["keep_recipe"]), ("linkhub", ["linkhub"]), ("both", ["keep_recipe", "linkhub"])):
    _nc_sub = _nc_d[_nc_d["app"].isin(_nc_apps) & _nc_d["variant"].isin(["baseline_debug", "allatori_debug"])]
    for _nc_w, _nc_col in T.WINDOWS.items():
        _nc_pr = T.pairs(_nc_sub, "allatori_debug", _nc_col)
        _nc_m, _nc_lo, _nc_hi, _ = T.ci_naive(_nc_pr["delta_pct"])
        _nc_rows.append({"negative_control": _nc_scope, "window": T.WINDOW_DISPLAY[_nc_w], "pairs": len(_nc_pr),
                     "cells_treated": int((_nc_sub["variant"] == "allatori_debug").sum()), "delta_pct": _nc_m, "ci_lo": _nc_lo, "ci_hi": _nc_hi,
                     "pairs_positive": int((_nc_pr["delta_pct"] > 0).sum()), "min_pair_pct": _nc_pr["delta_pct"].min(),
                     "max_pair_pct": _nc_pr["delta_pct"].max()})
_nc_tab = pd.DataFrame(_nc_rows)
_nc_tab.to_csv(T.TABLES / "negative_controls.csv", index=False)
T.register("tables/negative_controls.csv", "REPO", "4.3.2 Negative controls / Threats (noise floor)",
           "the two declared null builds (obfuscator did not run) against their own controls, matched pairs, both windows", NB)
print(_nc_tab.round(2).to_string(index=False))
for _nc_r in _nc_tab[_nc_tab["negative_control"] == "both"].itertuples():
    T.record(f"negative_controls.{_nc_r.window.lower()}", delta_pct=_nc_r.delta_pct, ci95=[_nc_r.ci_lo, _nc_r.ci_hi], pairs=int(_nc_r.pairs),
             method="paired t over matched pairs; exploratory, no p")

negative_control  window  pairs  cells_treated  delta_pct  ci_lo  ci_hi  pairs_positive  min_pair_pct  max_pair_pct
     keep_recipe  Active      6             60      -1.66  -3.88   0.57               1         -4.72          1.32
     keep_recipe Passive      6             60      -2.30  -7.79   3.18               2        -10.39          3.57
         linkhub  Active      9             89      -0.34  -4.13   3.45               5        -11.00          5.72
         linkhub Passive      9             89      -0.73  -4.19   2.74               5         -7.48          4.41
            both  Active     15            149      -0.87  -3.08   1.34               6        -11.00          5.72
            both Passive     15            149      -1.36  -3.96   1.24               7        -10.39          4.41


## Figures

Figure 1 shows what sits behind each headline number: every matched pair as a point, their spread as a box, and
the estimate with its 95% interval as a black diamond (`tables/pairs_delta.csv`, `tables/headline.csv`). The two panels
do **not** share an x-axis (workload spans a few percent, startup tens of percent). No titles; captions live in LaTeX.

**Every results figure reads against the control without its caption.** What is plotted is the matched-pair difference
from the unprotected control, so zero *is* the control: it is drawn as a labelled line, each axis says "vs. control",
and the control is never drawn as a bar or point of its own (`T.control_line`).

In [21]:
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
h = pd.read_csv(T.TABLES / "headline.csv")
pr = pd.read_csv(T.TABLES / "pairs_delta.csv")
techs = ["Bangcle", "Allatori", "freeRASP"]
yy = {t: k for t, k in zip(techs, (2, 1, 0))}            # Bangcle at the top, as in every other figure
jit = np.random.default_rng(0)                          # jitter for display only; no number depends on it
fig, axes = plt.subplots(1, 2, figsize=(T.FIG_SINGLE, 2.3), layout="constrained", sharey=True)
for ax, w, letter in zip(axes, ("workload", "startup"), "ab"):
    for t in techs:
        x = pr.loc[(pr["window"] == w) & (pr["technique"] == t), "delta_pct"].to_numpy()
        c, y0 = T.COLOURS[t], yy[t]
        ax.boxplot(x, positions=[y0 + 0.13], widths=0.32, orientation="horizontal", showfliers=False,
                   patch_artist=True, boxprops=dict(facecolor="none", edgecolor=c, linewidth=0.8),
                   medianprops=dict(color=c, linewidth=1.0), whiskerprops=dict(color=c, linewidth=0.8),
                   capprops=dict(color=c, linewidth=0.8))
        ax.scatter(x, y0 + 0.13 + jit.uniform(-0.1, 0.1, len(x)), s=2.5, color=c, alpha=0.5, linewidths=0, zorder=3)
        r = h[(h["window"] == w) & (h["technique"] == t)].iloc[0]
        ax.errorbar(r.delta_pct, y0 - 0.2, xerr=[[r.delta_pct - r.ci_lo], [r.ci_hi - r.delta_pct]], fmt="D",
                    color="black", ms=2.8, capsize=2, elinewidth=0.9, zorder=4)
    T.control_line(ax, "x", outside=True, label="unprotected\ncontrol")
    ax.set_xlabel(f"({letter}) {T.WINDOW_DISPLAY[w]}\n" + T.DELTA_LABEL)     # two lines: both labels fit one column
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(axis="y", length=0)
axes[0].set_yticks([yy[t] for t in techs]); axes[0].set_yticklabels(techs)
axes[0].set_ylim(-0.55, 2.5)
fig.legend(handles=[Patch(facecolor="none", edgecolor="0.3", label="matched pairs"),
                    Line2D([], [], marker="D", color="black", ms=2.8, label="mean, 95% CI")],
           loc="outside lower center", ncol=2, frameon=False, handlelength=1.2, columnspacing=1.5)
T.register(T.save_figure(fig, "figure1_headline", T.FIG_SINGLE), "REPO", "6.2 RQ1 (headline)",
           "every matched pair's delta-E vs. control (points, box) with the mean and 95% paired t CI, (a) active (b) passive", NB)

In [22]:
pa = pd.read_csv(T.TABLES / "per_app.csv")
apps_ = sorted(pa["app"].unique())
import matplotlib.transforms as mtransforms
fig, ax = plt.subplots(figsize=(T.FIG_SINGLE, 2.6), layout="constrained")
margin = mtransforms.blended_transform_factory(ax.transData, ax.transAxes)
for j, tech in enumerate(("Bangcle", "Allatori", "freeRASP")):
    d = pa[pa["technique"] == tech].set_index("app").reindex(apps_)
    x = np.arange(len(apps_)) + (j - 1) * 0.25
    ok = d["delta_pct"].notna().to_numpy()
    ax.scatter(x[ok], d["delta_pct"].to_numpy()[ok], s=9, color=T.COLOURS[tech], label=tech, zorder=3)
    if (~ok).any():
        # no estimate: marked in the margin under the axis, off the data scale, so it cannot be read as a value
        ax.scatter(x[~ok], np.full(int((~ok).sum()), -0.05), marker="x", s=12, linewidths=0.8,
                   color=T.COLOURS[tech], transform=margin, clip_on=False, label=f"{tech}: not analysed")
T.control_line(ax, "y", outside=True, label="unprotected\ncontrol")
ax.set_xticks(range(len(apps_))); ax.set_xticklabels(apps_, rotation=40, ha="right")
ax.tick_params(axis="x", pad=9)
ax.set_ylabel("Workload " + T.DELTA_LABEL)
fig.legend(frameon=False, ncol=4, loc="outside upper center", handletextpad=0.2, columnspacing=0.8)
ax.spines[["top", "right"]].set_visible(False)
T.register(T.save_figure(fig, "per_app_workload", T.FIG_SINGLE), "REPO", "6.2 RQ1 (per application)",
           "active-phase delta-E vs. control per application and technique; x under the axis marks an arm not analysed", NB)

In [23]:
ph = pd.read_csv(T.TABLES / "per_phone_freerasp.csv")
fig, ax = plt.subplots(figsize=(T.FIG_SINGLE, 1.9), layout="constrained")
x = np.arange(len(ph))
c = T.COLOURS["freeRASP"]
ax.errorbar(x - 0.1, ph["delta_pct"], yerr=[ph["delta_pct"] - ph["ci_lo"], ph["ci_hi"] - ph["delta_pct"]],
            fmt="o", color=c, capsize=2.5, ms=3.5, label="all cells (primary)")
ax.errorbar(x + 0.1, ph["filtered_delta_pct"], yerr=[ph["filtered_delta_pct"] - ph["filtered_ci_lo"], ph["filtered_ci_hi"] - ph["filtered_delta_pct"]],
            fmt="s", mfc="white", color=c, capsize=2.5, ms=3.5, label="post-charge cells removed")
T.control_line(ax, "y", at=0.01)
ax.set_xticks(x); ax.set_xticklabels(ph["phone"])
ax.set_ylabel("freeRASP active phase\n" + T.DELTA_LABEL); ax.legend(frameon=False, loc="upper right")
ax.spines[["top", "right"]].set_visible(False)
T.register(T.save_figure(fig, "per_phone_freerasp", T.FIG_SINGLE), "REPO", "6.2 RQ1 (per device)",
           "freeRASP active-phase delta-E per phone, primary and post-charge-filtered", NB)

## The headline-number file

`tables/headline_numbers.json` collects every headline number from all four notebooks (value, interval, p-value with its method, n), so the thesis text can be checked against it mechanically. Each notebook adds its own keys.

In [24]:
T.register("tables/headline_numbers.json", "REPO", "all of 4-6",
           "every headline number with interval, p-value (and method) and n; for mechanical checking of the thesis text", NB)
print(len(json.loads(T.HEADLINE_NUMBERS.read_text())), "keys")

64 keys
